# 06 – Chỉ mục FAISS 


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

CHUNK_DIR = PROJECT_ROOT / "knowledge" / "chunks"
CHUNKS_PATH = CHUNK_DIR / "chunks.jsonl"
INDEX_DIR = PROJECT_ROOT / "knowledge" / "index"
INDEX_PATH = INDEX_DIR / "faiss.index"
META_PATH = INDEX_DIR / "chunks_meta.json"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CHUNKS_PATH  =", CHUNKS_PATH, "exists=", CHUNKS_PATH.exists())

PROJECT_ROOT = d:\TTTN-VaxCare-2026\vaxcare-ai
CHUNKS_PATH  = d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl exists= True


## 1) Load chunks.jsonl 


In [2]:
def load_chunks(path: Path) -> list[dict]:
    rows = []
    skipped = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            c = json.loads(line)
            cid = str(c.get("chunk_id") or "")
            if "extra" in cid:
                skipped.append(cid)
                continue
            text = (c.get("text") or "").strip()
            if len(text) < 30:
                skipped.append(f"{cid}(short)")
                continue
            rows.append(c)
    if skipped:
        print("Skipped:", skipped)
    return rows

chunks = load_chunks(CHUNKS_PATH)


def load_yaml_chunks(chunk_dir: Path) -> list[dict]:
    """Nạp chunks_*.yaml (HDSD / WHO / TCMR) - đây là nguồn chunk_id của testset retrieval."""
    import yaml

    rows = []
    for f in sorted(chunk_dir.glob("chunks_*.yaml")):
        data = yaml.safe_load(f.read_text(encoding="utf-8"))
        for c in data:
            text = (c.get("text") or "").strip()
            if len(text) < 30:
                continue
            vid = c.get("vaccine_id")
            rows.append({
                "chunk_id": c["id"],
                "doc_id": c["id"].rsplit("-", 1)[0],
                "title": c.get("topic", ""),
                "topic": c.get("topic"),
                "text": text,
                "source_file": c.get("source_file"),
                "tier": c.get("source_id"),
                "vaccine_ids": [vid] if vid is not None else [],
                "page": c.get("source_pages"),
            })
        print(f"  {f.name}: {len(data)} chunks")
    return rows


yaml_chunks = load_yaml_chunks(CHUNK_DIR)
existing = {c["chunk_id"] for c in chunks}
chunks += [c for c in yaml_chunks if c["chunk_id"] not in existing]
print(f"Loaded {len(chunks)} chunks")
print("Sample IDs:", [c["chunk_id"] for c in chunks[:10]])
assert len(chunks) >= 70, "Quá ít chunk — kiểm tra lại chunks.jsonl"
assert not any("extra" in str(c.get("chunk_id", "")) for c in chunks), "Vẫn còn chunk extra!"

  chunks_hdsd_11_san_pham.yaml: 25 chunks
  chunks_tcmr_va_sang_loc.yaml: 18 chunks
  chunks_who_16_bai.yaml: 36 chunks
Loaded 449 chunks
Sample IDs: ['6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0000', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0001', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0002', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0003', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0004', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0005', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0006', '6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0007', '6381586785791217441575-qđ-byt-về

## 2) Encode bằng BGE-M3


In [3]:
from app.chatbot.rag.embedder import load_embedder

texts = [c.get("text") or "" for c in chunks]
embedder = load_embedder(prefer_bge=True, corpus_for_tfidf=texts)
print(f"Embedder: {getattr(embedder, 'model_name', '?')} dim={getattr(embedder, 'dim', '?')}")

print("Encoding...")
emb = embedder.encode(texts)
emb = np.asarray(emb, dtype=np.float32)
print("Embeddings shape:", emb.shape)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedder: BAAI/bge-m3 dim=1024
Encoding...


Inference Embeddings: 100%|██████████| 29/29 [15:50<00:00, 32.77s/it]  

Embeddings shape: (449, 1024)


## 3) Build FAISS + lưu index / meta / build_info


In [4]:
from app.chatbot.rag.vector_store import VectorStore

metas = []
for c in chunks:
    metas.append({
        "chunk_id": c.get("chunk_id"),
        "doc_id": c.get("doc_id"),
        "title": c.get("title"),
        "text": c.get("text"),
        "source_file": c.get("source_file"),
        "tier": c.get("tier"),
        "vaccine_ids": c.get("vaccine_ids") or [],
        "page": c.get("page"),
        "topic": c.get("topic"),
    })

store = VectorStore.build_from_embeddings(emb, metas, model_name=getattr(embedder, "model_name", ""))
print("index size", store.size, "dim", store.dim)

build_info = {
    "model": getattr(embedder, "model_name", ""),
    "dim": store.dim,
    "n_chunks": store.size,
    "built_at": datetime.now(timezone.utc).isoformat(),
    "chunks_path": str(CHUNKS_PATH),
    "sample_chunk_ids": [m["chunk_id"] for m in metas[:12]],
    "retrieval_min_score_suggested": 0.48,
    "retrieval_score_margin_suggested": 0.08,
    "note": "Built after removing *-extra-* boilerplate chunks",
}

INDEX_DIR.mkdir(parents=True, exist_ok=True)
store.save(INDEX_PATH, META_PATH, build_info=build_info)
print("Saved", INDEX_PATH)
print("Saved", META_PATH)
print("build_info:", json.dumps(build_info, ensure_ascii=False, indent=2))

index size 449 dim 1024
Saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\faiss.index
Saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\chunks_meta.json
build_info: {
  "model": "BAAI/bge-m3",
  "dim": 1024,
  "n_chunks": 449,
  "built_at": "2026-11-16T16:08:05.004390+00:00",
  "chunks_path": "d:\\TTTN-VaxCare-2026\\vaxcare-ai\\knowledge\\chunks\\chunks.jsonl",
  "sample_chunk_ids": [
    "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0000",
    "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0001",
    "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0002",
    "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0003",
    "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0004",
    "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-t

## 4) Smoke test retrieval

Ngưỡng production mới: `min_score=0.48`, `score_margin=0.08`.


In [5]:
from app.chatbot.rag.retriever import Retriever

retriever = Retriever(
    embedder=embedder,
    store=store,
    bm25=None,
    top_k=5,
    min_score=0.48,
    score_margin=0.08,
)

queries_pos = [
    "Trẻ sơ sinh tiêm những mũi nào theo TCMR?",
    "Vaccine HPV phòng bệnh gì?",
    "Chống chỉ định tiêm Varivax?",
    "Khám sàng lọc trước tiêm chủng trẻ em",
    "Vaccine BCG chỉ định cho đối tượng nào và liều lượng ra sao?",
]
queries_neg = [
    "Thời tiết Hà Nội tuần này thế nào?",
    "Cách nấu phở bò ngon",
    "Giá Bitcoin hôm nay",
    "Giá vaccine Gardasil 9 hiện nay bao nhiêu tiền?",
]

print("=== Positive (kỳ vọng enough_evidence=True) ===")
for q in queries_pos:
    r = retriever.retrieve(q)
    print(f"\nQ: {q}")
    print(f"  enough={r['enough_evidence']} max={r['max_score']:.3f} reason={r.get('rejected_reason')}")
    for h in r["hits"][:3]:
        snip = (h.get("text") or "")[:90].replace("\n", " ")
        print(f"  [{h['score']:.3f}] {h.get('chunk_id')} | {snip}")

print("\n=== Negative (kỳ vọng enough_evidence=False) ===")
neg_ok = 0
for q in queries_neg:
    r = retriever.retrieve(q)
    ok = not r["enough_evidence"]
    neg_ok += int(ok)
    print(f"Q: {q} → enough={r['enough_evidence']} max={r.get('raw_max_score', r['max_score']):.3f} reason={r.get('rejected_reason')} {'OK' if ok else 'FAIL'}")

print(f"\nneg_ok_rate = {neg_ok}/{len(queries_neg)} = {neg_ok/len(queries_neg):.2f}")

=== Positive (kỳ vọng enough_evidence=True) ===

Q: Trẻ sơ sinh tiêm những mũi nào theo TCMR?
  enough=True max=0.621 reason=None
  [0.621] tcmr-002 | Lịch cho trẻ sơ sinh theo TCMR: vắc xin viêm gan B đơn giá tiêm trong vòng 24 giờ sau sinh
  [0.610] tcmr-003 | Từ khi trẻ đủ 02 tháng tuổi, lịch TCMR gồm: vắc xin phối hợp có thành phần viêm gan B, vắc
  [0.600] tcmr-007 | Khi trẻ đủ 7 tuổi tiêm nhắc lại vắc xin phối hợp có thành phần bạch hầu giảm liều và vắc x

Q: Vaccine HPV phòng bệnh gì?
  enough=True max=0.707 reason=None
  [0.707] db-vaccine_11 | Vaccine: Vắc xin HPV (Human Papillomavirus) Nhà sản xuất: MSD – Hoa Kỳ Phòng bệnh: Ung thư
  [0.624] hpv-hdsd-001 | Theo tờ HDSD Gardasil 9 bản Việt Nam, vaccine chỉ định cho bé gái, phụ nữ và bé trai, nam 
  [0.581] db-protocol_12 | Phác đồ: Phác đồ HPV 3 liều Vaccine: Vắc xin HPV (Human Papillomavirus) Mô tả: Phác đồ 3 l

Q: Chống chỉ định tiêm Varivax?
  enough=True max=0.652 reason=None
  [0.652] bcg-hdsd-002 | Chống chỉ định với ngư